

## Semantic-Based Topic Modeling

In [1]:
from datetime import datetime
date = datetime.now()
formatted_date = date.strftime("%B %d, %Y")
print(formatted_date)

July 15, 2024


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
userdata.get('HF_TOKEN')

# Set up the current working directory within the Google Drive
%cd /content/drive/My\ Drive/Colab\ Notebooks/LLM/sped_biblio/topic_modeling

Mounted at /content/drive
/content/drive/My Drive/Colab Notebooks/LLM/sped_biblio/topic_modeling


In [3]:
# !pip install -q pandas numpy sentence-transformers bertopic scikit-learn matplotlib umap-learn hdbscan
!pip install -q qgrid nltk sentence_transformers bertopic umap-learn hdbscan dill networkx tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.2/889.2 kB 18.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 28.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 kB 21.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 20.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 50.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 71.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.1 MB/s eta 0:00:00


In [4]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='nltk')

import re
import warnings
from collections import defaultdict
import pickle
from pickle import UnpicklingError

# Data Manipulation
import dill
import numpy as np
import pandas as pd
import requests
import qgrid

# Natural Language Processing
import nltk
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertModel, BertTokenizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

# Clustering
from hdbscan import HDBSCAN
from umap import UMAP
from scipy.cluster import hierarchy as sch

# Visualization Imports
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.ticker import FuncFormatter
import colorlover as cl
import plotly.io as pio
import textwrap

# Network Analysis
import networkx as nx

# Progress Bar
from tqdm import tqdm

# Display HTML
from IPython.display import IFrame

nltk.download('wordnet')
pio.renderers.default = "colab"

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
[nltk_data] Downloading package wordnet to /root/nltk_data...


#### Combine text columns

In [5]:
all_data_file = f"files/all_data.xlsx"
all_data = pd.read_excel(all_data_file, na_filter=False)

df = all_data[all_data['filtered'] == 'Yes'].reset_index(drop=True)
df['Year'] = df['PY'].astype(int)
df['Decade'] = (df['Year'] // 10) * 10
df['CR'] = df['CR'].astype(str)

#### Cluster documents

In [6]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
sentence_embeddings = sentence_model.encode(df['combined_text'].tolist(), show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/102 [00:00<?, ?it/s]

In [7]:
umap_model = UMAP(
    n_neighbors=5,
    n_components=3,
    min_dist=0.01,
    metric='cosine',
    random_state=42
)

reduced_embeddings = umap_model.fit_transform(sentence_embeddings)

hdbscan_model = HDBSCAN(
    min_cluster_size=35,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

hdbscan_model.fit(reduced_embeddings)
labels = hdbscan_model.labels_

In [ ]:
df_cluster = pd.DataFrame(np.hstack([reduced_embeddings, labels.reshape(-1, 1)]),
     columns=["x", "y", "z", "cluster"]).sort_values("cluster")

# Visualize clusters
df_cluster['cluster'] = df_cluster['cluster'].astype(int).astype(str)

# Create interactive plot with Plotly
fig_cluster = px.scatter(df_cluster, x='x', y='y', color='cluster',
                 labels={'x': 'X', 'y': 'Y'},
                 hover_name='cluster')

fig_cluster.update_traces(marker=dict(size=7, opacity=0.3, line=dict(width=0.3, color="black")), selector=dict(mode='markers'))

fig_cluster.update_layout(
    title="<b>Clustered Documents</b>",
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    margin=dict(t=80, b=80, l=80, r=80),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=800,
    height=600,
    showlegend=True
)

fig_cluster.write_html("results/fig_cluster.html")
fig_cluster.show()

In [9]:
IFrame(src='results/fig_cluster.html', width=800, height=600)

#### Topic modeling

In [10]:
# Initialize and fit BERTopic
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(df['combined_text'])

In [ ]:
topic_model_fig = topic_model.visualize_topics()
topic_model_fig.update_layout(
    title="<b>Intertopic Distance Map</b>",
    title_x=0.55,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    width=800,
    height=600
)
topic_model_fig.write_html("results/topic_model_fig.html")
topic_model_fig.show()

In [12]:
IFrame(src='results/topic_model_fig.html', width=800, height=600)

In [13]:
lemmatizer = WordNetLemmatizer()
def lemmatize_text(text):
    tokens = text.split()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(lemmatized_tokens)

def preprocess_texts(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s-]', '', text)
    text = re.sub(r'\d+', '', text)
    tokens = text.split()
    return ' '.join(tokens)

df['lemmatized_text'] = df['combined_text'].apply(lemmatize_text)

df['preprocessed_text'] = df['lemmatized_text'].apply(preprocess_texts)

vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english')
ctfidf_model = ClassTfidfTransformer()
representation_model = KeyBERTInspired()

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model,
  representation_model=representation_model,
  calculate_probabilities=True,
  verbose=True
)

In [14]:
topics, probs = topic_model.fit_transform(df['preprocessed_text'])

2024-07-15 19:43:40,885 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/102 [00:00<?, ?it/s]

2024-07-15 19:43:43,553 - BERTopic - Embedding - Completed ✓
2024-07-15 19:43:43,555 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-07-15 19:43:57,764 - BERTopic - Dimensionality - Completed ✓
2024-07-15 19:43:57,766 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-07-15 19:43:58,027 - BERTopic - Cluster - Completed ✓
2024-07-15 19:43:58,032 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-07-15 19:44:03,846 - BERTopic - Representation - Completed ✓


In [ ]:
topic_info = topic_model.get_topic_info()

In [16]:
df['Topic'] = topics
for i in range(probs.shape[1]):
    df[f'Probability_Topic_{i}'] = probs[:, i]

In [70]:
print(df.columns.tolist())

['AU', 'AF', 'CR', 'AB', 'AR', 'BE', 'BN', 'BP', 'C1', 'C3', 'CL', 'CT', 'CY', 'DA', 'DE', 'DI', 'DT', 'EA', 'EF', 'EI', 'EM', 'EP', 'ER', 'FU', 'FX', 'GA', 'HC', 'HO', 'HP', 'ID', 'IS', 'J9', 'JI', 'LA', 'NR', 'OA', 'OI', 'PA', 'PD', 'PG', 'PI', 'PM', 'PN', 'PT', 'PU', 'PY', 'RI', 'RP', 'SC', 'SE', 'SI', 'SN', 'SO', 'SP', 'SU', 'TC', 'TI', 'U1', 'U2', 'UT', 'VL', 'WC', 'WE', 'Z9', 'C1raw', 'DB', 'AU_UN', 'AU1_UN', 'AU_UN_NR', 'SR_FULL', 'SR', 'CA', 'MA', 'GP', 'single_case', 'technology_use', 'combined_text', 'methodology', 'filtered', 'Year', 'Decade', 'lemmatized_text', 'preprocessed_text', 'Topic', 'Probability_Topic_0', 'Probability_Topic_1', 'Probability_Topic_2', 'Probability_Topic_3', 'Probability_Topic_4', 'Probability_Topic_5', 'Probability_Topic_6', 'Probability_Topic_7', 'Probability_Topic_8', 'Probability_Topic_9', 'Probability_Topic_10', 'Probability_Topic_11', 'Probability_Topic_12', 'Probability_Topic_13', 'Probability_Topic_14', 'Probability_Topic_15', 'Document', 'Nam

In [18]:
init_df = df.copy()
init_topic_info = topic_info.copy()

with open("files/topic_model.pkl", "wb") as f_model, \
     open("files/init_df.pkl", "wb") as f_init_df, \
     open("files/init_topic_info.pkl", "wb") as f_init_topic_info:

    pickle.dump(topic_model, f_model)
    pickle.dump(init_df, f_init_df)
    pickle.dump(init_topic_info, f_init_topic_info)

In [72]:
custom_labels = {
    -1: "Outlier",
    0: "Video modeling", 1: "Telehealth training", 2: "Functional analysis", 3: "Rehabilitation interventions",
    4: "Communication and speech", 5: "Mental health treatments", 6: "Reading instruction", 7: "Single-case data and analysis",
    8: "Behavior strategies", 9: "Mathematics instruction", 10: "Aphasia and auditory treatments", 11: "Applied behavior analysis",
    12: "Sports behavioral training", 13: "Staff training", 14: "Multiple disabilities", 15: "Self-monitoring"
}

topic_model.set_topic_labels(custom_labels)

In [73]:
vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english')
topic_model.update_topics(df['preprocessed_text'], vectorizer_model=vectorizer_model)

In [80]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,-1,660,-1_students_study_intervention_children,Outlier,"[students, study, intervention, children, part...",[TRAINING TEACHERS TO IMPLEMENT CLASSROOM PIVO...
1,0,471,0_video_skills_autism_social,Video modeling,"[video, skills, autism, social, modeling, vide...",[EFFECTS OF MOTHER-DELIVERED SOCIAL STORIES AN...
2,1,323,1_behavior_telehealth_aba_training,Telehealth training,"[behavior, telehealth, aba, training, parent, ...",[TELEHEALTH AND AUTISM TREATING CHALLENGING BE...
3,2,293,2_reinforcement_behavior_stereotypy_response,Functional analysis,"[reinforcement, behavior, stereotypy, response...",[A COMPARISON OF NONCONTINGENT REINFORCEMENT A...
4,3,212,3_design_intervention_study_memory,Rehabilitation interventions,"[design, intervention, study, memory, single, ...",[COMPUTER GAME-BASED UPPER EXTREMITY TRAINING ...
5,4,199,4_communication_speech_autism_children,Communication and speech,"[communication, speech, autism, children, aac,...",[A FURTHER COMPARISON OF MANUAL SIGNING PICTUR...
6,5,190,5_treatment_intervention_study_health,Mental health treatments,"[treatment, intervention, study, health, anxie...",[A MOBILE SELF-CONTROL TRAINING APP TO IMPROVE...
7,6,168,6_reading_students_words_instruction,Reading instruction,"[reading, students, words, instruction, interv...",[IPAD-ASSISTEDREADING FLUENCY INSTRUCTION FOR ...
8,7,150,7_single_data_case_single case,Single-case data and analysis,"[single, data, case, single case, analysis, vi...",[THE SINGLE-CASE REPORTING GUIDELINE IN BEHAVI...
9,8,117,8_behavior_students_gbg_disruptive,Behavior strategies,"[behavior, students, gbg, disruptive, classroo...",[EFFECTS OF THE GOOD BEHAVIOR GAME ON STUDENT ...


In [ ]:
topic_word_barchart = topic_model.visualize_barchart(top_n_topics=16, n_words=10, custom_labels=True)

traces = topic_word_barchart.data

num_charts = len(traces)
num_columns = 4
num_rows = (num_charts + num_columns - 1) // num_columns

fig = make_subplots(
    rows=num_rows,
    cols=num_columns,
    vertical_spacing=0.07,
    subplot_titles=[custom_labels[i] for i in range(num_charts)]
)

color_map = [
    "#5e4fa2", "#a799e8", "#1f78b4", "#9cd4f7",
    "#9cf4f7", "#33a02c", "#7CF3A0", "#66c2a5",
    "#abdda4", "#C8DF52", "#FAD02C", "#fdae61",
    "#fb9a99", "#f46d43", "#FF75D8", "#f53685"
]

df['Topic'] = topic_model.get_topic_info()['Topic']
color_map_extended = color_map + color_map[:max(0, len(custom_labels) - len(color_map))]
topic_color_map = {i: color for i, color in enumerate(color_map_extended[:len(custom_labels)])}

for i, trace in enumerate(traces):
    row = (i // num_columns) + 1
    col = (i % num_columns) + 1
    topic_idx = i

    if topic_idx == -1:
        trace.marker.color = 'darkgrey'
    else:
        trace.marker.color = topic_color_map[topic_idx]

    trace.update(width=0.8)
    trace.showlegend = False
    fig.add_trace(trace, row=row, col=col)

fig.update_layout(
    title_text="<b>Topic Word Scores</b>",
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    margin=dict(t=80, b=80, l=80, r=80),
    width=2000,
    height=1200,
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)

fig.write_html("results/topic_word_barchart.html")
fig.show()

In [132]:
IFrame(src='results/topic_word_barchart.html', width=2000, height=1200)

In [ ]:
titles = df['TI']

dff = pd.DataFrame(reduced_embeddings, columns=["x", "y", "z"])

dff["Topic"] = topics
dff["TI"] = titles

to_plot = dff.copy()
to_plot.loc[to_plot.Topic >= 16, "Topic"] = -1
outliers = to_plot.loc[to_plot.Topic == -1]
non_outliers = to_plot.loc[to_plot.Topic != -1]

to_plot['Topic'] = pd.to_numeric(to_plot['Topic'], errors='coerce')
to_plot['x'] = pd.to_numeric(to_plot['x'], errors='coerce')
to_plot['y'] = pd.to_numeric(to_plot['y'], errors='coerce')

to_plot = to_plot.dropna(subset=['Topic', 'x', 'y'])

non_outliers['color'] = non_outliers['Topic'].map(topic_color_map)

non_outliers['hover_color'] = non_outliers['color'].apply(lambda x: f'<span style="color:{x}">')

non_outliers_sorted = non_outliers.sort_values('Topic')

fig = go.Figure()

for topic in sorted(non_outliers_sorted['Topic'].unique()):
    topic_data = non_outliers_sorted[non_outliers_sorted['Topic'] == topic]
    topic_data['wrapped_TI'] = topic_data['TI'].apply(lambda text: "<br>".join(textwrap.wrap(text, width=50)))
    fig.add_trace(
        go.Scatter(
            x=topic_data['x'],
            y=topic_data['y'],
            mode='markers',
            marker=dict(
                # symbol='circle-open',
                size=7,
                line=dict(width=0.3, color="black"),
                color=topic_data['Topic'].map(topic_color_map),
                opacity=0.9
            ),
            text=topic_data['wrapped_TI'],
            hovertemplate='<b>%{text}</b><br><span style="font-size: 14px;">Topic: %{customdata}</span><extra></extra>',
            customdata=topic_data['Topic'].map(lambda t: custom_labels.get(t, 'Unknown')),
            name=custom_labels.get(topic, 'Unknown'),
            hoverinfo='text',
            showlegend=True
        )
    )

outliers['hovertext'] = outliers['Topic'].map(lambda t: "Topic: " + custom_labels.get(t, 'Unknown'))
outliers['wrapped_TI'] = outliers['TI'].apply(lambda text: "<br>".join(textwrap.wrap(text, width=50)))
fig.add_scatter(
    x=outliers['x'],
    y=outliers['y'],
    mode='markers',
    marker=dict(
        # symbol='circle-open',
        line=dict(width=0.3, color="black"),
        color='#e8e8e8',
        size=7,
        opacity=0.7
    ),
    name='Outliers',
    text=outliers['wrapped_TI'],
            hovertemplate='<b>%{text}</b><br><span style="font-size: 14px;">Topic: %{customdata}</span><extra></extra>',
    customdata=outliers['Topic'].map(lambda t: custom_labels.get(t, 'Unknown')),
    hoverinfo='text'
)

fig.update_layout(
    hoverlabel=dict(
        font_size=12,
        namelength=-1
    )
)

centroids = to_plot.groupby("Topic")[['x', 'y']].mean().reset_index().iloc[1:]
annotation_positions = []
for index, row in centroids.iterrows():
    topic = int(row.Topic)
    # text = f"<b>{topic}:</b> " + "_".join([x[0] for x in topic_model.get_topic(topic)[:3]])
    text = f"Topic: {topic}"
    x_offset = 0
    y_offset = 0
    position = (row.x + x_offset, row.y + y_offset)
    while position in annotation_positions:
        x_offset += 0.1
        y_offset += 0.05
        position = (row.x + x_offset, row.y + y_offset)

    fig.add_annotation(
        x=position[0],
        y=position[1],
        text=text,
        showarrow=False,
        font=dict(size=13),
        xanchor='center',
        visible=False
    )
    annotation_positions.append(position)

fig.update_layout(
    title="<b>Documents and Topics</b>",
    title_x=0.5,
    title_y=0.99,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    margin=dict(t=10, b=10, l=80, r=80),
    width=1000,
    height=600,
    showlegend=True,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False),
    legend=dict(traceorder='normal')
)

fig.write_html("results/docs_topics.html")
fig.show()

In [142]:
IFrame(src='results/docs_topics.html', width=1000, height=600)

In [ ]:
heatmap = topic_model.visualize_heatmap(custom_labels=True)
heatmap.update_layout(
    title="<b>Similarity Matrix</b>",
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    width=900,
    height=800
)
heatmap.write_html("results/heatmap.html")
heatmap.show()

In [144]:
IFrame(src='results/heatmap.html', width=900, height=800)

In [ ]:
# Hierarchical topics
linkage_function = lambda x: sch.linkage(x, 'single', optimal_ordering=True)
hierarchical_topics = topic_model.hierarchical_topics(df['preprocessed_text'], linkage_function=linkage_function)
hierarchical_topics_fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics, custom_labels=True)

hierarchical_topics_fig.update_layout(
    title="<b>Hierarchical Clustering</b>",
    title_x=0.49,
    title_font=dict(size=24, family="Helvetica Neue", color="black"),
    width=800,
    height=600
)

hierarchical_topics_fig.write_html("results/hierarchical_topics_fig.html")
hierarchical_topics_fig.show()

In [146]:
IFrame(src='results/hierarchical_topics_fig.html', width=800, height=600)

In [105]:
def extract_ngrams(X, features):
    unigrams = []
    bigrams = []
    trigrams = []

    for row in X:
        present_ngrams = features[row.indices]
        unigrams.append([token for token in present_ngrams if len(token.split()) == 1])
        bigrams.append([token for token in present_ngrams if len(token.split()) == 2])
        trigrams.append([token for token in present_ngrams if len(token.split()) == 3])

    return unigrams, bigrams, trigrams

In [106]:
doc_info = topic_model.get_document_info(df['preprocessed_text'])
doc_info_df = pd.DataFrame(doc_info)
df = pd.concat([df.reset_index(drop=True), doc_info_df], axis=1)

vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english')
ngram_matrix = vectorizer_model.fit_transform(df['preprocessed_text'])
features = np.array(vectorizer_model.get_feature_names_out())
df['unigrams'], df['bigrams'], df['trigrams'] = extract_ngrams(ngram_matrix, features)

In [107]:
def extract_ngrams_per_topic(df, ngram_type='unigrams'):
    ngrams_per_topic = defaultdict(lambda: defaultdict(set))
    for custom_label in df['CustomName'].unique():
        topic_df = df[df['CustomName'] == custom_label]
        for Year in sorted(topic_df['Year'].unique()):
            year_df = topic_df[topic_df['Year'] == Year]
            for ngrams in year_df[ngram_type]:
                ngrams_per_topic[custom_label][Year].update(ngrams)
    return ngrams_per_topic

def calculate_proportion_new_ngrams_per_topic(ngrams_per_topic):
    proportions = []
    for CustomName, years_ngrams in ngrams_per_topic.items():
        previous_ngrams = set()
        for Year, ngrams in sorted(years_ngrams.items()):
            new_unique_ngrams = ngrams - previous_ngrams
            proportion_new_ngrams = len(new_unique_ngrams) / len(ngrams) if ngrams else 0
            previous_ngrams = previous_ngrams.union(ngrams)
            proportions.append({
                'CustomName': CustomName,
                'Year': Year,
                'unique_ngram_count': len(ngrams),
                'proportion_new_ngrams': proportion_new_ngrams
            })
    return pd.DataFrame(proportions)

def comma_formatter(value, _):
    return f"{value:,}"

def decimal_formatter(value, _):
    return f"{value:.2f}"

In [ ]:
df['Topic'] = topics
df['CustomName'] = df['Topic'].map(custom_labels)

In [110]:
if 'CustomName' in df.columns:
    df = df.loc[:, ~df.columns.duplicated()]

df_filtered = df[df['Topic'] != -1].reset_index(drop=True)

df['CustomName'] = pd.Categorical(df['CustomName'], categories=sorted(df['CustomName'].unique()), ordered=True)

In [113]:
unigrams_per_topic = extract_ngrams_per_topic(df_filtered, 'unigrams')
bigrams_per_topic = extract_ngrams_per_topic(df_filtered, 'bigrams')
trigrams_per_topic = extract_ngrams_per_topic(df_filtered, 'trigrams')

unigram_proportions_topic = calculate_proportion_new_ngrams_per_topic(unigrams_per_topic)
bigram_proportions_topic = calculate_proportion_new_ngrams_per_topic(bigrams_per_topic)
trigram_proportions_topic = calculate_proportion_new_ngrams_per_topic(trigrams_per_topic)

unigram_proportions_topic['ngram_type'] = 'unigram'
bigram_proportions_topic['ngram_type'] = 'bigram'
trigram_proportions_topic['ngram_type'] = 'trigram'

all_proportions_topic = pd.concat([unigram_proportions_topic, bigram_proportions_topic, trigram_proportions_topic])

In [ ]:
all_proportions_topic['Year'] = all_proportions_topic['Year'].astype(str)

colors = {
    'unigram': '#ff9d00',
    'bigram': '#1F78B4',
    'trigram': '#6A3D9A',
}

unique_topics = df_filtered.sort_values('Topic')['CustomName'].unique()
num_cols = 4
num_rows = (len(unique_topics) // num_cols) + (1 if len(unique_topics) % num_cols != 0 else 0)

min_y = all_proportions_topic['unique_ngram_count'].min()
max_y = all_proportions_topic['unique_ngram_count'].max()

ngram_count_fig = make_subplots(
    rows=num_rows, cols=num_cols, subplot_titles=unique_topics,
    vertical_spacing=0.1, horizontal_spacing=0.1
)

legend_added = {'unigram': False, 'bigram': False, 'trigram': False}
for i, custom_label in enumerate(unique_topics):
    topic_df = all_proportions_topic[all_proportions_topic['CustomName'] == custom_label]
    row = (i // num_cols) + 1
    col = (i % num_cols) + 1
    for ngram_type in ['unigram', 'bigram', 'trigram']:
        filtered_df = topic_df[topic_df['ngram_type'] == ngram_type]
        ngram_count_fig.add_trace(
            go.Scatter(
                x=filtered_df['Year'],
                y=filtered_df['unique_ngram_count'],
                mode='lines+markers',
                name=ngram_type.capitalize() if not legend_added[ngram_type] else None,
                line=dict(color=colors[ngram_type], width=0.9),
                marker=dict(size=3),
                hovertemplate=(
                    'N-gram Type: %{text}<br>'
                    'Year: %{x}<br>'
                    'Custom Label: %{customdata}<br>'
                    'Count: %{y}<br>'
                ),
                text=filtered_df['ngram_type'],
                customdata=[custom_label] * len(filtered_df),
                showlegend=not legend_added[ngram_type]
            ),
            row=row,
            col=col
        )
        legend_added[ngram_type] = True

ngram_count_fig.update_layout(
    width=num_cols * 400,
    height=num_rows * 250,
    showlegend=True,
    legend_title_text='N-gram Type',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.1,
        xanchor="center",
        x=0.5
    ),
    title_text="<b>Unique N-gram Counts Over Years by Topic</b>",
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black")
)

ngram_count_fig.add_annotation(
    x=-0.08, y=0.3,
    xref='paper', yref='paper',
    showarrow=False,
    textangle=-90,
    text="Unique N-grams"
)

ngram_count_fig.update_xaxes(tickangle=-70, showline=True, linewidth=0.7, linecolor='black', showticklabels=True)
ngram_count_fig.update_yaxes(showline=True, linewidth=0.7, linecolor='black', showticklabels=True, title_text=None)

ngram_count_fig.write_html("results/ngram_count_fig.html")
ngram_count_fig.show()

In [115]:
IFrame(src='results/ngram_count_fig.html', width=1450, height=1000)

In [ ]:
ngram_proportion_fig = make_subplots(
    rows=num_rows, cols=num_cols, subplot_titles=unique_topics,
    vertical_spacing=0.1, horizontal_spacing=0.1
)

legend_added = {'unigram': False, 'bigram': False, 'trigram': False}
for i, custom_label in enumerate(unique_topics):
    topic_df = all_proportions_topic[all_proportions_topic['CustomName'] == custom_label]
    row = (i // num_cols) + 1
    col = (i % num_cols) + 1
    for ngram_type in ['unigram', 'bigram', 'trigram']:
        filtered_df = topic_df[topic_df['ngram_type'] == ngram_type]
        ngram_proportion_fig.add_trace(
            go.Scatter(
                x=filtered_df['Year'],
                y=filtered_df['proportion_new_ngrams'],
                mode='lines+markers',
                name=ngram_type.capitalize() if not legend_added[ngram_type] else None,
                line=dict(color=colors[ngram_type], width=0.9, dash='dot'),
                marker=dict(size=3),
                hovertemplate=(
                    'N-gram Type: %{text}<br>'
                    'Year: %{x}<br>'
                    'Custom Label: %{customdata}<br>'
                    'Proportion: %{y}<br>'
                ),
                text=filtered_df['ngram_type'],
                customdata=[custom_label] * len(filtered_df),
                showlegend=not legend_added[ngram_type]
            ),
            row=row,
            col=col
        )
        legend_added[ngram_type] = True

ngram_proportion_fig.update_layout(
    width=num_cols * 400,
    height=num_rows * 250,
    showlegend=True,
    legend_title_text='N-gram Type',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.1,
        xanchor="center",
        x=0.5
    ),
    title_text="<b>Unique N-gram Proportions Over Years by Topic</b>",
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    title_x=0.5,
    title_font=dict(size=24, family="Helvetica Neue", color="black")
)

ngram_proportion_fig.add_annotation(
    x=-0.08, y=0.3,
    xref='paper', yref='paper',
    showarrow=False,
    text="New N-grams/Unique N-grams",
    textangle=-90
)

ngram_proportion_fig.update_xaxes(tickangle=-70, showline=True, linewidth=0.7, linecolor='black', showticklabels=True)
ngram_proportion_fig.update_yaxes(showline=True, linewidth=0.7, linecolor='black', showticklabels=True, title_text=None)

ngram_proportion_fig.write_html("results/ngram_proportion_fig.html")
ngram_proportion_fig.show()

In [117]:
IFrame(src='results/ngram_proportion_fig.html', width=1450, height=1000)

In [118]:
with open("files/topic_model.pkl", "wb") as f_model, \
     open("files/df.pkl", "wb") as f_df, \
     open("files/topic_info.pkl", "wb") as f_topic_info:

    pickle.dump(topic_model, f_model)
    pickle.dump(df, f_df)
    pickle.dump(topic_info, f_topic_info)

In [119]:
with pd.ExcelWriter('files/df.xlsx') as writer:
    df.to_excel(writer, sheet_name='Sheet1', index=False)

with pd.ExcelWriter('files/topic_info.xlsx') as writer:
    topic_info.to_excel(writer, sheet_name='Sheet1', index=False)

In [40]:
# with open("files/df.pkl", "rb") as f_df, \
#      open("files/topic_info.pkl", "rb") as f_topic_info:

#     df = pickle.load(f_df)
#     topic_info = pickle.load(f_topic_info)

In [41]:
# df = pd.read_excel('files/df.xlsx')
# topic_info = pd.read_excel('files/topic_info.xlsx')

In [147]:
from nbconvert import HTMLExporter
import nbformat

notebook_path = 'index.ipynb'
html_exporter = HTMLExporter()

with open(notebook_path, 'r', encoding='utf-8') as nb_file:
    notebook_content = nb_file.read()
    notebook = nbformat.reads(notebook_content, as_version=4)

html_output, _ = html_exporter.from_notebook_node(notebook)

with open('index.html', 'w', encoding='utf-8') as html_file:
    html_file.write(html_output)